# Phần 5 - Mô hình phân loại dựa trên LSTM/GRU

**Mục tiêu**: Xây dựng mô hình phân loại ảnh MNIST sử dụng LSTM và GRU bằng cách biểu diễn ảnh thành chuỗi (sequence).

**Dataset**: MNIST (28×28 grayscale images, 10 classes)

**Các cách biểu diễn sequence**:
1. **Row-wise**: Mỗi hàng của ảnh là một timestep (seq_len=28, features=28)
2. **Column-wise**: Mỗi cột của ảnh là một timestep (seq_len=28, features=28)
3. **Patch-based**: Chia ảnh thành patches (seq_len=16, features=49)

**Mô hình**: 
- LSTM với 3 cách biểu diễn
- GRU với 3 cách biểu diễn
- **Tổng cộng**: 6 models để so sánh

---
## 1. Import thư viện và thiết lập môi trường

In [ ]:
# Import các thư viện cần thiết
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split

import torchvision
from torchvision import datasets, transforms

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import time
import os

from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

# Thiết lập style cho plots
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✓ Import thành công!")
print(f"PyTorch version: {torch.__version__}")

### 1.1 Thiết lập seed cho reproducibility

In [ ]:
# Set seed cho reproducibility
SEED = 42

torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Kiểm tra device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

### 1.2 Định nghĩa hyperparameters

In [ ]:
# Hyperparameters
BATCH_SIZE = 128
NUM_EPOCHS = 15
LEARNING_RATE = 0.001
HIDDEN_SIZE = 128
NUM_LAYERS = 2
NUM_CLASSES = 10
DROPOUT = 0.2

# Sequence-specific parameters
SEQUENCE_CONFIG = {
    'row': {'input_size': 28, 'seq_length': 28},
    'column': {'input_size': 28, 'seq_length': 28},
    'patch': {'input_size': 49, 'seq_length': 16}  # 4x4 patches, 7x7 pixels each
}

print("Hyperparameters:")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Epochs: {NUM_EPOCHS}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Hidden size: {HIDDEN_SIZE}")
print(f"  Num layers: {NUM_LAYERS}")
print(f"\nSequence configurations:")
for method, config in SEQUENCE_CONFIG.items():
    print(f"  {method}: seq_len={config['seq_length']}, input_size={config['input_size']}")

---
## 2. Tải và chuẩn bị dữ liệu MNIST

In [ ]:
# Định nghĩa transforms
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))  # MNIST mean và std
])

# Tải MNIST dataset
train_dataset = datasets.MNIST(
    root='./data',
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.MNIST(
    root='./data',
    train=False,
    download=True,
    transform=transform
)

# Chia train thành train/val (50000/10000)
train_size = 50000
val_size = len(train_dataset) - train_size
train_dataset, val_dataset = random_split(
    train_dataset, 
    [train_size, val_size],
    generator=torch.Generator().manual_seed(SEED)
)

# Tạo DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True if torch.cuda.is_available() else False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True if torch.cuda.is_available() else False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True if torch.cuda.is_available() else False
)

print(f"✓ Dataset loaded successfully!")
print(f"  Train: {len(train_dataset)} samples")
print(f"  Val: {len(val_dataset)} samples")
print(f"  Test: {len(test_dataset)} samples")
print(f"  Batches per epoch: {len(train_loader)}")

### 2.1 Visualize dữ liệu mẫu

In [ ]:
# Lấy một batch để visualize
sample_images, sample_labels = next(iter(train_loader))

print(f"Batch shape: {sample_images.shape}")  # (batch_size, 1, 28, 28)
print(f"Labels shape: {sample_labels.shape}")  # (batch_size,)
print(f"Data type: {sample_images.dtype}")
print(f"Value range: [{sample_images.min():.3f}, {sample_images.max():.3f}]")

# Visualize 10 ảnh đầu tiên
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
fig.suptitle('MNIST Sample Images', fontsize=14, fontweight='bold')

for idx, ax in enumerate(axes.flat):
    img = sample_images[idx].squeeze().numpy()
    label = sample_labels[idx].item()
    
    ax.imshow(img, cmap='gray')
    ax.set_title(f'Label: {label}', fontsize=11)
    ax.axis('off')

plt.tight_layout()
plt.show()

---
## 3. Định nghĩa các hàm biến đổi sequence

Chúng ta sẽ triển khai 3 cách biểu diễn ảnh thành chuỗi để đưa vào LSTM/GRU.

In [ ]:
def row_wise_sequence(images):
    """
    Biểu diễn ảnh theo hàng: mỗi hàng là một timestep.
    
    Args:
        images: Tensor shape (batch, 1, 28, 28)
    
    Returns:
        Tensor shape (batch, 28, 28)
        - 28 timesteps (mỗi hàng)
        - 28 features (giá trị pixel trong mỗi hàng)
    """
    return images.squeeze(1)  # Remove channel dimension


def column_wise_sequence(images):
    """
    Biểu diễn ảnh theo cột: mỗi cột là một timestep.
    
    Args:
        images: Tensor shape (batch, 1, 28, 28)
    
    Returns:
        Tensor shape (batch, 28, 28)
        - 28 timesteps (mỗi cột)
        - 28 features (giá trị pixel trong mỗi cột)
    """
    images = images.squeeze(1)  # (batch, 28, 28)
    return images.transpose(1, 2)  # Transpose height và width


def patch_based_sequence(images, patch_size=7):
    """
    Biểu diễn ảnh theo patches: chia ảnh thành các ô vuông.
    
    Args:
        images: Tensor shape (batch, 1, 28, 28)
        patch_size: Kích thước mỗi patch (default=7 cho ảnh 28x28)
    
    Returns:
        Tensor shape (batch, 16, 49)
        - 16 timesteps (4x4 = 16 patches)
        - 49 features (7x7 = 49 pixels mỗi patch)
    """
    batch_size = images.size(0)
    images = images.squeeze(1)  # (batch, 28, 28)
    
    # Sử dụng unfold để extract patches
    # unfold(dimension, size, step)
    patches = images.unfold(1, patch_size, patch_size).unfold(2, patch_size, patch_size)
    # Shape: (batch, 4, 4, 7, 7)
    
    # Reshape thành sequence
    patches = patches.contiguous().view(batch_size, -1, patch_size * patch_size)
    # Shape: (batch, 16, 49)
    
    return patches


print("✓ Sequence transformation functions defined!")

### 3.1 Test và visualize các transformations

In [ ]:
# Test với 1 ảnh
test_img = sample_images[0:1]  # (1, 1, 28, 28)
test_label = sample_labels[0].item()

print(f"Original image shape: {test_img.shape}")
print(f"Label: {test_label}\n")

# Test các transformations
row_seq = row_wise_sequence(test_img)
col_seq = column_wise_sequence(test_img)
patch_seq = patch_based_sequence(test_img)

print(f"Row-wise sequence shape: {row_seq.shape}")      # (1, 28, 28)
print(f"Column-wise sequence shape: {col_seq.shape}")   # (1, 28, 28)
print(f"Patch-based sequence shape: {patch_seq.shape}")  # (1, 16, 49)

# Visualize
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle(f'Sequence Transformation Visualization (Label: {test_label})', 
             fontsize=14, fontweight='bold')

# Original
axes[0, 0].imshow(test_img.squeeze().numpy(), cmap='gray')
axes[0, 0].set_title('Original Image\n(28×28)', fontweight='bold')
axes[0, 0].axis('off')

# Row-wise
axes[0, 1].imshow(row_seq.squeeze().numpy(), cmap='viridis', aspect='auto')
axes[0, 1].set_title('Row-wise Sequence\n(28 steps × 28 features)', fontweight='bold')
axes[0, 1].set_xlabel('Features (pixel values)')
axes[0, 1].set_ylabel('Timesteps (rows)')

# Column-wise
axes[0, 2].imshow(col_seq.squeeze().numpy(), cmap='viridis', aspect='auto')
axes[0, 2].set_title('Column-wise Sequence\n(28 steps × 28 features)', fontweight='bold')
axes[0, 2].set_xlabel('Features (pixel values)')
axes[0, 2].set_ylabel('Timesteps (columns)')

# Patch-based
axes[0, 3].imshow(patch_seq.squeeze().numpy(), cmap='viridis', aspect='auto')
axes[0, 3].set_title('Patch-based Sequence\n(16 steps × 49 features)', fontweight='bold')
axes[0, 3].set_xlabel('Features (pixels per patch)')
axes[0, 3].set_ylabel('Timesteps (patches)')

# Visualize từng timestep đầu tiên
axes[1, 0].axis('off')

# First row
first_row = row_seq[0, 0, :].numpy()
axes[1, 1].plot(first_row, linewidth=2)
axes[1, 1].set_title('First Row (timestep 0)', fontsize=10)
axes[1, 1].set_xlabel('Pixel index')
axes[1, 1].set_ylabel('Pixel value')
axes[1, 1].grid(True, alpha=0.3)

# First column
first_col = col_seq[0, 0, :].numpy()
axes[1, 2].plot(first_col, linewidth=2, color='orange')
axes[1, 2].set_title('First Column (timestep 0)', fontsize=10)
axes[1, 2].set_xlabel('Pixel index')
axes[1, 2].set_ylabel('Pixel value')
axes[1, 2].grid(True, alpha=0.3)

# First patch
first_patch = patch_seq[0, 0, :].numpy().reshape(7, 7)
axes[1, 3].imshow(first_patch, cmap='gray')
axes[1, 3].set_title('First Patch (7×7)', fontsize=10)
axes[1, 3].axis('off')

plt.tight_layout()
plt.show()

print("\n✓ Visualization complete!")

---
## 4. Định nghĩa kiến trúc mô hình

Chúng ta sẽ xây dựng 2 classes: `LSTMClassifier` và `GRUClassifier`.

In [ ]:
class LSTMClassifier(nn.Module):
    """
    LSTM classifier cho phân loại ảnh MNIST.
    
    Args:
        input_size: Số features mỗi timestep
        hidden_size: Số neurons trong hidden state
        num_layers: Số LSTM layers
        num_classes: Số classes đầu ra
        sequence_method: Cách biểu diễn sequence ('row', 'column', 'patch')
        dropout: Dropout rate (chỉ áp dụng nếu num_layers > 1)
    """
    
    def __init__(self, input_size, hidden_size, num_layers, num_classes, 
                 sequence_method='row', dropout=0.2):
        super(LSTMClassifier, self).__init__()
        
        self.sequence_method = sequence_method
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        # LSTM layer
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )
        
        # Fully connected layer
        self.fc = nn.Linear(hidden_size, num_classes)
    
    def forward(self, x):
        """
        Forward pass.
        
        Args:
            x: Input tensor shape (batch, 1, 28, 28)
        
        Returns:
            Output logits shape (batch, num_classes)
        """
        # Transform image to sequence
        if self.sequence_method == 'row':
            x = row_wise_sequence(x)
        elif self.sequence_method == 'column':
            x = column_wise_sequence(x)
        elif self.sequence_method == 'patch':
            x = patch_based_sequence(x)
        else:
            raise ValueError(f"Unknown sequence method: {self.sequence_method}")
        
        # LSTM forward
        # out: (batch, seq_len, hidden_size)
        # h_n: (num_layers, batch, hidden_size)
        # c_n: (num_layers, batch, hidden_size)
        out, (h_n, c_n) = self.lstm(x)
        
        # Lấy output của timestep cuối cùng
        out = out[:, -1, :]  # (batch, hidden_size)
        
        # Classification head
        out = self.fc(out)  # (batch, num_classes)
        
        return out


class GRUClassifier(nn.Module):
    """
    GRU classifier cho phân loại ảnh MNIST.
    
    Tương tự LSTMClassifier nhưng sử dụng GRU thay vì LSTM.
    """
    
    def __init__(self, input_size, hidden_size, num_layers, num_classes,
                 sequence_method='row', dropout=0.2):
        super(GRUClassifier, self).__init__()
        
        self.sequence_method = sequence_method
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        # GRU layer
        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )
        
        # Fully connected layer
        self.fc = nn.Linear(hidden_size, num_classes)
    
    def forward(self, x):
        """
        Forward pass.
        
        Args:
            x: Input tensor shape (batch, 1, 28, 28)
        
        Returns:
            Output logits shape (batch, num_classes)
        """
        # Transform image to sequence
        if self.sequence_method == 'row':
            x = row_wise_sequence(x)
        elif self.sequence_method == 'column':
            x = column_wise_sequence(x)
        elif self.sequence_method == 'patch':
            x = patch_based_sequence(x)
        else:
            raise ValueError(f"Unknown sequence method: {self.sequence_method}")
        
        # GRU forward
        # out: (batch, seq_len, hidden_size)
        # h_n: (num_layers, batch, hidden_size)
        out, h_n = self.gru(x)
        
        # Lấy output của timestep cuối cùng
        out = out[:, -1, :]  # (batch, hidden_size)
        
        # Classification head
        out = self.fc(out)  # (batch, num_classes)
        
        return out


print("✓ Model classes defined!")

### 4.1 Test model và hiển thị architecture

In [ ]:
# Test LSTM model với row-wise sequence
test_model = LSTMClassifier(
    input_size=SEQUENCE_CONFIG['row']['input_size'],
    hidden_size=HIDDEN_SIZE,
    num_layers=NUM_LAYERS,
    num_classes=NUM_CLASSES,
    sequence_method='row',
    dropout=DROPOUT
).to(device)

# Forward pass test
test_input = torch.randn(2, 1, 28, 28).to(device)
test_output = test_model(test_input)

print("LSTM Model Test:")
print(f"  Input shape: {test_input.shape}")
print(f"  Output shape: {test_output.shape}")
print(f"  Expected: (2, {NUM_CLASSES})")
print(f"  ✓ Forward pass successful!\n")

# Đếm số parameters
total_params = sum(p.numel() for p in test_model.parameters())
trainable_params = sum(p.numel() for p in test_model.parameters() if p.requires_grad)

print(f"Model Architecture:")
print(test_model)
print(f"\nParameters:")
print(f"  Total: {total_params:,}")
print(f"  Trainable: {trainable_params:,}")

# Cleanup
del test_model, test_input, test_output
torch.cuda.empty_cache() if torch.cuda.is_available() else None

---
## 5. Training và Evaluation Functions

In [ ]:
def train_one_epoch(model, train_loader, criterion, optimizer, device, epoch):
    """
    Train model cho 1 epoch.
    
    Returns:
        avg_loss, accuracy
    """
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{NUM_EPOCHS} [Train]')
    
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)
        
        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # Backward pass và optimization
        optimizer.zero_grad()
        loss.backward()
        
        # Gradient clipping để tránh exploding gradients
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        
        # Statistics
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        
        # Update progress bar
        pbar.set_postfix({
            'loss': f'{running_loss/(pbar.n+1):.4f}',
            'acc': f'{100.*correct/total:.2f}%'
        })
    
    avg_loss = running_loss / len(train_loader)
    accuracy = 100. * correct / total
    
    return avg_loss, accuracy


def evaluate(model, data_loader, criterion, device, mode='Val'):
    """
    Evaluate model trên validation hoặc test set.
    
    Returns:
        avg_loss, accuracy, predictions, labels
    """
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        pbar = tqdm(data_loader, desc=f'[{mode}]')
        
        for images, labels in pbar:
            images, labels = images.to(device), labels.to(device)
            
            # Forward pass
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            # Statistics
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            
            # Store predictions
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
            # Update progress bar
            pbar.set_postfix({
                'loss': f'{running_loss/(pbar.n+1):.4f}',
                'acc': f'{100.*correct/total:.2f}%'
            })
    
    avg_loss = running_loss / len(data_loader)
    accuracy = 100. * correct / total
    
    return avg_loss, accuracy, all_preds, all_labels


def train_model(model, model_name, train_loader, val_loader, 
                criterion, optimizer, device, num_epochs):
    """
    Training loop đầy đủ cho một model.
    
    Returns:
        history dict với train/val loss và accuracy
    """
    print(f"\n{'='*60}")
    print(f"Training: {model_name}")
    print(f"{'='*60}")
    
    history = {
        'train_loss': [],
        'train_acc': [],
        'val_loss': [],
        'val_acc': []
    }
    
    best_val_acc = 0.0
    start_time = time.time()
    
    for epoch in range(num_epochs):
        # Training
        train_loss, train_acc = train_one_epoch(
            model, train_loader, criterion, optimizer, device, epoch
        )
        
        # Validation
        val_loss, val_acc, _, _ = evaluate(
            model, val_loader, criterion, device, mode='Val'
        )
        
        # Save history
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        
        # Print epoch summary
        print(f"\nEpoch {epoch+1}/{num_epochs}:")
        print(f"  Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%")
        print(f"  Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%")
        
        # Save best model
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            print(f"  ✓ New best validation accuracy: {best_val_acc:.2f}%")
    
    training_time = time.time() - start_time
    print(f"\nTraining completed in {training_time:.2f}s")
    print(f"Best validation accuracy: {best_val_acc:.2f}%")
    
    return history, training_time


print("✓ Training functions defined!")

---
## 6. Huấn luyện các mô hình LSTM

Chúng ta sẽ train 3 LSTM models với 3 cách biểu diễn sequence khác nhau.

### 6.1 LSTM Row-wise

In [ ]:
# Khởi tạo model
lstm_row = LSTMClassifier(
    input_size=SEQUENCE_CONFIG['row']['input_size'],
    hidden_size=HIDDEN_SIZE,
    num_layers=NUM_LAYERS,
    num_classes=NUM_CLASSES,
    sequence_method='row',
    dropout=DROPOUT
).to(device)

# Loss và optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(lstm_row.parameters(), lr=LEARNING_RATE)

# Train
history_lstm_row, time_lstm_row = train_model(
    lstm_row, 'LSTM Row-wise',
    train_loader, val_loader,
    criterion, optimizer, device, NUM_EPOCHS
)

### 6.2 LSTM Column-wise

In [ ]:
# Khởi tạo model
lstm_col = LSTMClassifier(
    input_size=SEQUENCE_CONFIG['column']['input_size'],
    hidden_size=HIDDEN_SIZE,
    num_layers=NUM_LAYERS,
    num_classes=NUM_CLASSES,
    sequence_method='column',
    dropout=DROPOUT
).to(device)

# Loss và optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(lstm_col.parameters(), lr=LEARNING_RATE)

# Train
history_lstm_col, time_lstm_col = train_model(
    lstm_col, 'LSTM Column-wise',
    train_loader, val_loader,
    criterion, optimizer, device, NUM_EPOCHS
)

### 6.3 LSTM Patch-based

In [ ]:
# Khởi tạo model
lstm_patch = LSTMClassifier(
    input_size=SEQUENCE_CONFIG['patch']['input_size'],
    hidden_size=HIDDEN_SIZE,
    num_layers=NUM_LAYERS,
    num_classes=NUM_CLASSES,
    sequence_method='patch',
    dropout=DROPOUT
).to(device)

# Loss và optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(lstm_patch.parameters(), lr=LEARNING_RATE)

# Train
history_lstm_patch, time_lstm_patch = train_model(
    lstm_patch, 'LSTM Patch-based',
    train_loader, val_loader,
    criterion, optimizer, device, NUM_EPOCHS
)

---
## 7. Huấn luyện các mô hình GRU

Tương tự, chúng ta sẽ train 3 GRU models.

### 7.1 GRU Row-wise

In [ ]:
# Khởi tạo model
gru_row = GRUClassifier(
    input_size=SEQUENCE_CONFIG['row']['input_size'],
    hidden_size=HIDDEN_SIZE,
    num_layers=NUM_LAYERS,
    num_classes=NUM_CLASSES,
    sequence_method='row',
    dropout=DROPOUT
).to(device)

# Loss và optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(gru_row.parameters(), lr=LEARNING_RATE)

# Train
history_gru_row, time_gru_row = train_model(
    gru_row, 'GRU Row-wise',
    train_loader, val_loader,
    criterion, optimizer, device, NUM_EPOCHS
)

### 7.2 GRU Column-wise

In [ ]:
# Khởi tạo model
gru_col = GRUClassifier(
    input_size=SEQUENCE_CONFIG['column']['input_size'],
    hidden_size=HIDDEN_SIZE,
    num_layers=NUM_LAYERS,
    num_classes=NUM_CLASSES,
    sequence_method='column',
    dropout=DROPOUT
).to(device)

# Loss và optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(gru_col.parameters(), lr=LEARNING_RATE)

# Train
history_gru_col, time_gru_col = train_model(
    gru_col, 'GRU Column-wise',
    train_loader, val_loader,
    criterion, optimizer, device, NUM_EPOCHS
)

### 7.3 GRU Patch-based

In [ ]:
# Khởi tạo model
gru_patch = GRUClassifier(
    input_size=SEQUENCE_CONFIG['patch']['input_size'],
    hidden_size=HIDDEN_SIZE,
    num_layers=NUM_LAYERS,
    num_classes=NUM_CLASSES,
    sequence_method='patch',
    dropout=DROPOUT
).to(device)

# Loss và optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(gru_patch.parameters(), lr=LEARNING_RATE)

# Train
history_gru_patch, time_gru_patch = train_model(
    gru_patch, 'GRU Patch-based',
    train_loader, val_loader,
    criterion, optimizer, device, NUM_EPOCHS
)

---
## 8. Đánh giá trên Test Set

Sau khi train xong, chúng ta sẽ evaluate tất cả 6 models trên test set.

In [ ]:
# Dictionary chứa tất cả models
all_models = {
    'LSTM Row-wise': lstm_row,
    'LSTM Column-wise': lstm_col,
    'LSTM Patch-based': lstm_patch,
    'GRU Row-wise': gru_row,
    'GRU Column-wise': gru_col,
    'GRU Patch-based': gru_patch
}

all_histories = {
    'LSTM Row-wise': history_lstm_row,
    'LSTM Column-wise': history_lstm_col,
    'LSTM Patch-based': history_lstm_patch,
    'GRU Row-wise': history_gru_row,
    'GRU Column-wise': history_gru_col,
    'GRU Patch-based': history_gru_patch
}

all_times = {
    'LSTM Row-wise': time_lstm_row,
    'LSTM Column-wise': time_lstm_col,
    'LSTM Patch-based': time_lstm_patch,
    'GRU Row-wise': time_gru_row,
    'GRU Column-wise': time_gru_col,
    'GRU Patch-based': time_gru_patch
}

# Evaluate trên test set
test_results = {}
criterion = nn.CrossEntropyLoss()

print("\n" + "="*60)
print("EVALUATING ON TEST SET")
print("="*60)

for model_name, model in all_models.items():
    print(f"\nEvaluating {model_name}...")
    test_loss, test_acc, test_preds, test_labels = evaluate(
        model, test_loader, criterion, device, mode='Test'
    )
    
    # Đếm parameters
    num_params = sum(p.numel() for p in model.parameters())
    
    test_results[model_name] = {
        'test_loss': test_loss,
        'test_acc': test_acc,
        'predictions': test_preds,
        'labels': test_labels,
        'num_params': num_params,
        'training_time': all_times[model_name]
    }
    
    print(f"  Test Loss: {test_loss:.4f}")
    print(f"  Test Accuracy: {test_acc:.2f}%")
    print(f"  Parameters: {num_params:,}")
    print(f"  Training Time: {all_times[model_name]:.2f}s")

print("\n✓ All models evaluated!")

---
## 9. So sánh và Visualization

### 9.1 Bảng so sánh tổng hợp

In [ ]:
# Tạo DataFrame so sánh
comparison_data = []

for model_name, results in test_results.items():
    history = all_histories[model_name]
    
    comparison_data.append({
        'Model': model_name,
        'Test Accuracy (%)': f"{results['test_acc']:.2f}",
        'Test Loss': f"{results['test_loss']:.4f}",
        'Best Val Accuracy (%)': f"{max(history['val_acc']):.2f}",
        'Parameters': f"{results['num_params']:,}",
        'Training Time (s)': f"{results['training_time']:.2f}"
    })

comparison_df = pd.DataFrame(comparison_data)

print("\n" + "="*80)
print("SO SÁNH TỔNG HỢP CÁC MÔ HÌNH")
print("="*80)
print(comparison_df.to_string(index=False))
print("="*80)

# Tìm model tốt nhất
best_model_idx = comparison_df['Test Accuracy (%)'].astype(float).idxmax()
best_model_name = comparison_df.loc[best_model_idx, 'Model']
best_accuracy = comparison_df.loc[best_model_idx, 'Test Accuracy (%)']

print(f"\n🏆 Best Model: {best_model_name} với Test Accuracy = {best_accuracy}%")

### 9.2 Biểu đồ so sánh Test Accuracy

In [ ]:
# Bar chart so sánh test accuracy
fig, ax = plt.subplots(figsize=(12, 6))

models = list(test_results.keys())
accuracies = [test_results[m]['test_acc'] for m in models]

# Tô màu khác nhau cho LSTM và GRU
colors = ['steelblue' if 'LSTM' in m else 'coral' for m in models]

bars = ax.bar(range(len(models)), accuracies, color=colors, alpha=0.8, edgecolor='black')

# Highlight best model
best_idx = accuracies.index(max(accuracies))
bars[best_idx].set_color('gold')
bars[best_idx].set_edgecolor('darkred')
bars[best_idx].set_linewidth(3)

# Thêm giá trị trên mỗi bar
for i, (bar, acc) in enumerate(zip(bars, accuracies)):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{acc:.2f}%',
            ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_xlabel('Model', fontsize=12, fontweight='bold')
ax.set_ylabel('Test Accuracy (%)', fontsize=12, fontweight='bold')
ax.set_title('So sánh Test Accuracy của các mô hình LSTM/GRU', 
             fontsize=14, fontweight='bold')
ax.set_xticks(range(len(models)))
ax.set_xticklabels(models, rotation=45, ha='right')
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_ylim([min(accuracies) - 2, 100])

# Legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='steelblue', edgecolor='black', label='LSTM'),
    Patch(facecolor='coral', edgecolor='black', label='GRU'),
    Patch(facecolor='gold', edgecolor='darkred', linewidth=3, label='Best')
]
ax.legend(handles=legend_elements, loc='lower right')

plt.tight_layout()
plt.show()

### 9.3 Training curves comparison

In [ ]:
# Plot training curves cho tất cả models
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Training History Comparison', fontsize=16, fontweight='bold')

# Define colors
lstm_colors = ['blue', 'cyan', 'navy']
gru_colors = ['red', 'orange', 'darkred']

# Plot 1: Train Loss
ax = axes[0, 0]
for idx, (model_name, history) in enumerate(all_histories.items()):
    color = lstm_colors[idx % 3] if 'LSTM' in model_name else gru_colors[idx % 3]
    ax.plot(history['train_loss'], label=model_name, linewidth=2, color=color)
ax.set_xlabel('Epoch', fontsize=11)
ax.set_ylabel('Loss', fontsize=11)
ax.set_title('Training Loss', fontweight='bold')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Plot 2: Validation Loss
ax = axes[0, 1]
for idx, (model_name, history) in enumerate(all_histories.items()):
    color = lstm_colors[idx % 3] if 'LSTM' in model_name else gru_colors[idx % 3]
    ax.plot(history['val_loss'], label=model_name, linewidth=2, 
            linestyle='--', color=color)
ax.set_xlabel('Epoch', fontsize=11)
ax.set_ylabel('Loss', fontsize=11)
ax.set_title('Validation Loss', fontweight='bold')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Plot 3: Train Accuracy
ax = axes[1, 0]
for idx, (model_name, history) in enumerate(all_histories.items()):
    color = lstm_colors[idx % 3] if 'LSTM' in model_name else gru_colors[idx % 3]
    ax.plot(history['train_acc'], label=model_name, linewidth=2, color=color)
ax.set_xlabel('Epoch', fontsize=11)
ax.set_ylabel('Accuracy (%)', fontsize=11)
ax.set_title('Training Accuracy', fontweight='bold')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Plot 4: Validation Accuracy
ax = axes[1, 1]
for idx, (model_name, history) in enumerate(all_histories.items()):
    color = lstm_colors[idx % 3] if 'LSTM' in model_name else gru_colors[idx % 3]
    ax.plot(history['val_acc'], label=model_name, linewidth=2, 
            linestyle='--', color=color)
ax.set_xlabel('Epoch', fontsize=11)
ax.set_ylabel('Accuracy (%)', fontsize=11)
ax.set_title('Validation Accuracy', fontweight='bold')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 9.4 So sánh theo nhóm

In [ ]:
# So sánh LSTM vs GRU với cùng sequence method
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('LSTM vs GRU Comparison (cùng sequence method)', 
             fontsize=14, fontweight='bold')

sequence_methods = ['Row-wise', 'Column-wise', 'Patch-based']

for idx, method in enumerate(sequence_methods):
    ax = axes[idx]
    
    lstm_name = f'LSTM {method}'
    gru_name = f'GRU {method}'
    
    lstm_acc = test_results[lstm_name]['test_acc']
    gru_acc = test_results[gru_name]['test_acc']
    
    bars = ax.bar(['LSTM', 'GRU'], [lstm_acc, gru_acc], 
                   color=['steelblue', 'coral'], alpha=0.8, edgecolor='black')
    
    # Add values
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.2f}%',
                ha='center', va='bottom', fontsize=11, fontweight='bold')
    
    ax.set_ylabel('Test Accuracy (%)', fontsize=11)
    ax.set_title(f'{method} Sequence', fontweight='bold')
    ax.set_ylim([min(lstm_acc, gru_acc) - 2, 100])
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

### 9.5 Confusion Matrix cho model tốt nhất

In [ ]:
# Tìm best model
best_model_name = max(test_results.items(), 
                      key=lambda x: x[1]['test_acc'])[0]
best_results = test_results[best_model_name]

# Tính confusion matrix
cm = confusion_matrix(best_results['labels'], best_results['predictions'])

# Plot
fig, ax = plt.subplots(figsize=(12, 10))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=range(10), yticklabels=range(10),
            cbar_kws={'label': 'Count'}, ax=ax)

ax.set_xlabel('Predicted Label', fontsize=12, fontweight='bold')
ax.set_ylabel('True Label', fontsize=12, fontweight='bold')
ax.set_title(f'Confusion Matrix - {best_model_name}\nTest Accuracy: {best_results["test_acc"]:.2f}%',
             fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

# Classification report
print(f"\nClassification Report - {best_model_name}:")
print("="*60)
print(classification_report(best_results['labels'], best_results['predictions'],
                          target_names=[str(i) for i in range(10)]))

---
## 10. Phân tích và Nhận xét

### 10.1 So sánh các cách biểu diễn Sequence

**Quan sát từ kết quả thực nghiệm trên Kaggle (Tesla T4 GPU):**

1. **Row-wise Sequence - BEST PERFORMER** ⭐:
   - **LSTM Row-wise**: 99.02% test accuracy
   - **GRU Row-wise**: 99.10% test accuracy (highest!)
   - **Tại sao tốt nhất?**:
     - Cách biểu diễn tự nhiên nhất cho ảnh chữ số
     - Mỗi hàng chứa thông tin liên tục về hình dạng chữ số
     - Phù hợp với cách con người đọc và nhận diện chữ (từ trái sang phải, từ trên xuống)
     - Sequence length hợp lý (28 timesteps) - không quá dài, không quá ngắn

2. **Column-wise Sequence - MODERATE**:
   - **LSTM Column-wise**: 98.65% test accuracy
   - **GRU Column-wise**: 98.46% test accuracy
   - **Phân tích**:
     - Kém hơn row-wise khoảng 0.4-0.6%
     - Vẫn đạt accuracy cao (>98%)
     - Cột không mang thông tin liên tục về chữ số tự nhiên như hàng
     - Tuy nhiên, với ảnh đối xứng như MNIST, column-wise vẫn hoạt động tốt

3. **Patch-based Sequence - WORST PERFORMER** ⚠️:
   - **LSTM Patch-based**: 98.21% test accuracy
   - **GRU Patch-based**: 98.05% test accuracy (lowest!)
   - **Tại sao kém nhất?**:
     - **Mất thông tin global structure**: Chia ảnh thành patches làm mất continuity
     - **Sequence quá ngắn** (16 timesteps): LSTM/GRU không tận dụng được khả năng model long-range dependencies
     - **Input dimension lớn hơn** (49 vs 28): Nhiều parameters hơn nhưng không cải thiện performance
     - **Overfitting**: Training accuracy cao (99.5%+) nhưng validation/test accuracy thấp hơn
     - Patch-based phù hợp hơn cho CNN hoặc Vision Transformer, không phải RNN

**Key Insights:**

✅ **Row-wise là best choice cho LSTM/GRU trên image classification**
- Preserves natural structure của ảnh
- Sequence length tối ưu cho MNIST (28 steps)
- Đơn giản và hiệu quả

❌ **Patch-based không phù hợp với RNN**
- Sequence quá ngắn
- Mất thông tin spatial continuity
- Nhiều parameters nhưng kém hiệu quả

📊 **Practical Recommendation**:
Nếu phải dùng RNN cho image classification, hãy dùng **row-wise sequence** với cấu trúc đơn giản nhất.

### 10.2 So sánh LSTM vs GRU

**Quan sát từ kết quả thực nghiệm trên Kaggle:**

**So sánh Performance theo từng sequence method:**

| Sequence Method | LSTM Accuracy | GRU Accuracy | Winner | Difference |
|-----------------|---------------|--------------|--------|------------|
| **Row-wise** | 99.02% | **99.10%** | GRU | +0.08% |
| **Column-wise** | **98.65%** | 98.46% | LSTM | +0.19% |
| **Patch-based** | **98.21%** | 98.05% | LSTM | +0.16% |

**Overall: GRU Row-wise là BEST model (99.10%)**

---

**1. Performance Analysis:**
   - **GRU Row-wise đạt highest accuracy**: 99.10%
   - LSTM Row-wise rất gần: 99.02% (chỉ kém 0.08%)
   - Với row-wise (best sequence method), **GRU nhỉnh hơn LSTM**
   - Với column-wise và patch-based, LSTM nhỉnh hơn GRU nhưng cả hai đều kém hơn row-wise

**2. Parameters Comparison:**
   - **GRU ít parameters hơn đáng kể**:
     - LSTM Row/Column-wise: **214,282 parameters**
     - GRU Row/Column-wise: **161,034 parameters** (25% ít hơn)
     - LSTM Patch-based: **225,034 parameters**
     - GRU Patch-based: **169,098 parameters** (25% ít hơn)
   - **GRU hiệu quả hơn**: Ít parameters hơn nhưng performance tương đương hoặc tốt hơn
   - **Memory efficient**: GRU tiết kiệm bộ nhớ hơn, phù hợp cho deployment

**3. Training Time Comparison:**
   - Cả 6 models train trên GPU Tesla T4:
     - LSTM: ~123-125 giây/model
     - GRU: ~120-124 giây/model
   - **GRU nhanh hơn khoảng 2-4%**
   - Với training time tương đương, GRU có advantage về efficiency

**4. Convergence Behavior:**
   - **GRU converge nhanh hơn**:
     - GRU Row-wise đạt 98.78% val accuracy ở epoch 7
     - LSTM Row-wise đạt 98.60% val accuracy ở epoch 6
   - **GRU stable hơn**:
     - Validation curves của GRU ít fluctuate hơn
     - LSTM có xu hướng overfit nhẹ ở epochs cuối

**5. Tại sao GRU tốt hơn LSTM trên MNIST?**
   - **Simpler architecture**: 2 gates (reset, update) vs 3 gates (forget, input, output)
   - **Less prone to overfitting**: Ít parameters hơn → generalize tốt hơn
   - **MNIST là dataset đơn giản**: Không cần complexity của LSTM
   - **Shorter sequences (28 steps)**: GRU đủ khả năng handle, không cần cell state riêng của LSTM

---

**Key Takeaways:**

✅ **GRU là lựa chọn tốt hơn cho MNIST** (và các image classification tasks tương tự):
- Highest accuracy (99.10%)
- 25% ít parameters hơn LSTM
- Training nhanh hơn
- Memory efficient

✅ **LSTM vẫn có giá trị** khi:
- Sequences rất dài (>100 timesteps)
- Cần model complex temporal dependencies
- Dataset phức tạp hơn MNIST

📊 **Practical Recommendation**:
Với image classification using RNN, ưu tiên **GRU với row-wise sequence** - đơn giản, hiệu quả, và performance tốt nhất.

### 10.3 So sánh với các mô hình khác

**LSTM/GRU vs CNN/Transformer cho Image Classification:**

**1. Performance Comparison trên MNIST:**

| Model Type | Best Accuracy | Parameters | Training Time | Inference Speed |
|------------|---------------|------------|---------------|-----------------|
| **CNN (State-of-the-art)** | ~99.7%+ | ~50K-100K | Very Fast | Very Fast |
| **Vision Transformer (ViT)** | ~98-99% | ~1M-10M | Medium | Medium |
| **RNN (LSTM/GRU) - Ours** | **99.10%** | ~161K-225K | Slow | Slow |
| **Simple MLP** | ~97-98% | ~100K | Very Fast | Very Fast |

**Kết quả của chúng ta (99.10%) là IMPRESSIVE cho RNN-based approach!**

---

**2. Tại sao CNN vẫn tốt hơn cho ảnh?**

**CNN Advantages:**
- ✅ **Spatial Inductive Bias**: CNN architecture được thiết kế cho 2D images
  - Convolution preserves spatial relationships
  - Local connectivity matches image structure
  - Translation invariance: Detect features ở bất kỳ vị trí nào

- ✅ **Parameter Efficiency**: 
  - Shared weights across spatial locations
  - Ít parameters hơn nhưng performance tốt hơn
  - Example: LeNet-5 chỉ ~60K params đạt 99%+

- ✅ **Parallel Processing**:
  - Tất cả pixels/features được process đồng thời
  - Rất nhanh trên GPU
  - Scalable cho real-time applications

**RNN Disadvantages cho images:**
- ❌ **Sequential Processing**: Phải process từng timestep tuần tự
- ❌ **Mất 2D Structure**: Flatten ảnh → mất spatial relationships
- ❌ **Slow**: Không thể parallelize như CNN
- ❌ **Vanishing Gradient**: Vẫn có issues với sequences dài

---

**3. Khi nào nên dùng LSTM/GRU cho ảnh?**

✅ **Hợp lý khi:**
- **Video Analysis**: Temporal sequence of frames
  - Action recognition trong videos
  - Video classification/segmentation
  
- **Image Captioning**: Combine CNN + RNN
  - CNN extract visual features
  - RNN generate text descriptions
  
- **Sequential Image Generation**: 
  - PixelRNN/PixelCNN
  - Generate images pixel-by-pixel
  
- **Temporal Image Data**:
  - Medical imaging sequences (CT scans over time)
  - Satellite imagery time series

❌ **KHÔNG nên dùng cho:**
- Static image classification thuần túy (như MNIST, CIFAR-10)
- Real-time applications (quá chậm)
- Cases cần exploit spatial structure

---

**4. Computational Cost Analysis:**

**Training Time (15 epochs trên MNIST):**
- CNN: ~30-60 giây (estimate)
- Our RNN: ~120-125 giây (actual)
- **RNN chậm hơn 2-4x**

**Inference Speed:**
- CNN: ~0.001 ms/image
- RNN: ~0.01 ms/image
- **RNN chậm hơn ~10x**

**Memory Footprint:**
- CNN: Low (parallel processing)
- RNN: Higher (sequential states)

---

**5. Hybrid Approaches (Best of Both Worlds):**

**CNN + RNN Combinations:**
1. **CNN-LSTM**: 
   - CNN extracts spatial features
   - LSTM processes feature sequences
   - Use case: Video analysis, image captioning

2. **ConvLSTM**:
   - LSTM with convolutional operations
   - Preserves spatial structure
   - Use case: Video prediction, weather forecasting

3. **Attention Mechanisms**:
   - Focus on important spatial/temporal regions
   - Reduces RNN limitations
   - Use case: Image captioning with attention

---

**Key Insights:**

📊 **Kết quả 99.10% của chúng ta cho thấy**:
- RNN CÓ THỂ đạt performance tốt trên image classification
- NHƯNG không phải optimal choice
- Trade-off: High accuracy vs Computational cost

🎯 **Practical Recommendations**:
1. **For static images**: Dùng CNN (faster, simpler, better)
2. **For videos/temporal data**: Dùng RNN hoặc CNN+RNN hybrid
3. **For image captioning**: CNN (encoder) + RNN (decoder)
4. **For research/learning**: RNN trên images là bài tập tốt để hiểu sequence modeling!

💡 **Bài học quan trọng**:
RNN đạt 99.10% không phải vì nó là "best tool" cho images, mà vì:
- MNIST là dataset đơn giản
- 15 epochs training với GPU tốt
- Careful hyperparameter tuning
- Optimal sequence representation (row-wise)

**Trong production**: Dùng CNN cho speed và efficiency!

### 10.4 Limitations và Future Work

**Limitations của phương pháp này:**

1. **Mất thông tin 2D spatial structure**:
   - Ảnh được flatten thành 1D sequence
   - Mất relationships giữa các pixels theo 2D
   - CNN preserve spatial structure tốt hơn

2. **Sequential processing inefficiency**:
   - Không thể parallelize như CNN
   - Training và inference chậm hơn
   - Không phù hợp cho real-time applications

3. **Vanishing gradient** (dù đã cải thiện với LSTM/GRU):
   - Vẫn còn vấn đề với sequences dài (28 timesteps)
   - Early timesteps khó học

**Hướng cải thiện:**

1. **Hybrid approaches**:
   - CNN + LSTM/GRU: CNN extract features, RNN process sequence
   - ConvLSTM: Combine convolution với LSTM cells

2. **Bidirectional LSTM/GRU**:
   - Process sequence cả forward và backward
   - Capture context tốt hơn

3. **Attention mechanism**:
   - Focus vào important parts của sequence
   - Giảm dependency vào sequence length

4. **Better sequence representation**:
   - Spiral scanning
   - Hilbert curve
   - Multi-scale patches

### 10.5 Kết luận

**Tóm tắt kết quả thực nghiệm trên Kaggle (Tesla T4 GPU):**

---

## 🏆 Best Model: **GRU Row-wise - 99.10% Test Accuracy**

**Ranking đầy đủ:**
1. 🥇 **GRU Row-wise**: 99.10% (161K params, 123.81s)
2. 🥈 **LSTM Row-wise**: 99.02% (214K params, 124.58s)
3. 🥉 **LSTM Column-wise**: 98.65% (214K params, 124.72s)
4. **GRU Column-wise**: 98.46% (161K params, 122.28s)
5. **LSTM Patch-based**: 98.21% (225K params, 123.77s)
6. **GRU Patch-based**: 98.05% (169K params, 120.61s)

---

## 📊 Key Findings:

### 1️⃣ **Sequence Representation Method** (Most Important Factor)

**Row-wise >> Column-wise >> Patch-based**

- ✅ **Row-wise tốt nhất**: 99.02-99.10% accuracy
  - Preserves natural structure của chữ số
  - Sequence length optimal (28 timesteps)
  - Mỗi hàng chứa thông tin liên tục
  
- ⚠️ **Column-wise moderate**: 98.46-98.65% accuracy
  - Vẫn hoạt động tốt nhưng kém row-wise
  - Column không natural như row cho digit recognition
  
- ❌ **Patch-based worst**: 98.05-98.21% accuracy  
  - Sequence quá ngắn (16 steps)
  - Mất global structure information
  - Overfitting (train acc 99.5%+ but test acc lowest)

**Unexpected Discovery**: Ban đầu nghĩ patch-based sẽ tốt nhất (như ViT), nhưng kết quả cho thấy **simplicity wins** cho RNN!

---

### 2️⃣ **LSTM vs GRU**

**GRU nhỉnh hơn với best sequence method:**

| Metric | LSTM | GRU | Winner |
|--------|------|-----|--------|
| **Best Accuracy** | 99.02% | **99.10%** | GRU |
| **Parameters** | 214K | **161K** | GRU (-25%) |
| **Training Time** | 124.58s | **123.81s** | GRU |
| **Memory** | Higher | **Lower** | GRU |

**Tại sao GRU tốt hơn?**
- Simpler architecture (2 gates vs 3 gates)
- Less overfitting (fewer parameters)
- MNIST không cần complexity của LSTM
- Sequence ngắn (28 steps) → GRU sufficient

---

### 3️⃣ **Performance Analysis**

**So với các mô hình khác trên MNIST:**
- CNN State-of-the-art: ~99.7%+
- **Our GRU Row-wise: 99.10%** ✅
- Vision Transformer: ~98-99%
- Simple MLP: ~97-98%

**Impressive result!** RNN đạt gần SOTA mặc dù:
- ❌ Không có spatial inductive bias
- ❌ Sequential processing (slow)
- ❌ Phải flatten 2D → 1D

---

## 💡 Key Insights & Lessons Learned:

### ✅ **What Worked:**

1. **Simple is Better**: Row-wise sequence (simplest) >> Patch-based (complex)
2. **GRU > LSTM**: Cho simple datasets, lighter model often better
3. **Proper Hyperparameters**:
   - Hidden size: 128 (sufficient)
   - Num layers: 2 (not too deep)
   - Dropout: 0.2 (prevents overfitting)
   - Learning rate: 0.001 (stable)
4. **Gradient Clipping**: Essential để avoid exploding gradients
5. **GPU Training**: Tesla T4 cho phép train nhanh (~2 phút/model)

### ❌ **What Didn't Work:**

1. **Patch-based Sequence**: Không phù hợp với RNN
   - Tốt cho ViT, nhưng RNN cần longer sequences
2. **Over-complexity**: More parameters ≠ Better performance
3. **Column-wise**: Natural order matters for sequential models

---

## 🎯 Practical Recommendations:

### **Khi nào dùng RNN cho images?**

✅ **Hợp lý:**
- Video analysis (temporal sequences)
- Image captioning (CNN encoder + RNN decoder)
- Sequential image generation
- Medical imaging time series

❌ **KHÔNG nên:**
- Static image classification → **Dùng CNN**
- Real-time applications → **Dùng CNN**  
- Production systems → **Dùng CNN** (faster)

### **Nếu PHẢI dùng RNN cho images:**
1. ✅ Chọn **GRU** (lighter, faster, equally good)
2. ✅ Dùng **Row-wise sequence** (simplest, best)
3. ✅ Keep architecture simple (2 layers, hidden=128)
4. ✅ Use gradient clipping
5. ✅ Train with GPU

---

## 🎓 Educational Value:

**Bài tập này có giá trị cao về mặt học tập:**

1. **Hiểu sâu về Sequence Modeling**:
   - Cách RNN process sequential data
   - LSTM vs GRU architecture differences
   - Vanishing/exploding gradient issues

2. **Nhận ra Inductive Bias quan trọng**:
   - CNN has spatial bias → perfect for images
   - RNN has sequential bias → not ideal for images
   - Right tool for right job!

3. **Hands-on Implementation**:
   - Self-written training loops
   - Forward → Loss → Backward → Step
   - Not just calling high-level APIs

4. **Critical Thinking**:
   - Challenge assumptions (patch-based ≠ best)
   - Analyze why simpler method works better
   - Understand trade-offs (accuracy vs speed vs complexity)

---

## 🚀 Final Thoughts:

**Câu hỏi ban đầu**: "RNN có thể classify images không?"

**Câu trả lời**: 
- ✅ **CÓ THỂ** - và đạt 99.10% accuracy!
- ⚠️ **NHƯNG KHÔNG NÊN** - CNN tốt hơn về mọi mặt (speed, efficiency, performance)

**Bài học lớn nhất**:
> "Just because you CAN doesn't mean you SHOULD. Understanding WHY to use each architecture is more important than knowing HOW to implement them."

RNN brilliant cho temporal data (text, speech, videos).  
CNN brilliant cho spatial data (images).  
**Use the right tool for the job!** 🔧

---

**Cảm ơn đã đọc đến đây! Hy vọng bài thực hành này giúp bạn hiểu sâu hơn về LSTM, GRU, và image classification.** 🎉

---
**End of Notebook**